<a href="https://colab.research.google.com/github/Ashu-42/quant_research_crypto_volatility_forecasting/blob/quant_DL_ashu/notebooks/09_Rolling30_Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 09 — Rolling-30-Day Historical Volatility: Data Processing

This notebook creates the alternative robustness target:

\[
HV_{30,t} =
\operatorname{Std}(r_{t-29},\ldots,r_t)
\]

using 30 consecutive daily close-to-close log returns and `ddof=1`.

For model training and QLIKE evaluation, the variance-scale target is

\[
HV^2_{30,t}
\]

and the deep-learning target is

\[
\log(HV^2_{30,t}+\epsilon).
\]

### Methodology retained from the primary experiment

- The same finalized daily return series is reused.
- Forecast horizon remains one day ahead.
- Train / validation / test periods remain unchanged.
- Scaling is fitted on training observations only.
- Sequence continuity checks are retained.
- 7d / 14d / 30d BTC artifacts are created so the selected DL specifications can be retrained.
- Existing squared-daily-return artifacts are never overwritten.

### Robustness-specific feature design

The feature count remains 7. The three variance-state features are replaced by rolling-30 historical-variance analogues:

1. `log_rolling_30d_variance`
2. `daily_return`
3. `absolute_daily_return`
4. `high_low_range`
5. `log_volume`
6. `rolling30_var_mean_7d`
7. `rolling30_var_mean_30d`

This gives the DL models the current rolling-volatility state while preserving the same broad feature structure as the primary experiment.

## 1. Imports and paths

In [1]:
from google.colab import drive
from sklearn.preprocessing import StandardScaler
from pathlib import Path

import json
import joblib
import numpy as np
import pandas as pd

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Quant Research"
)

SOURCE_DAILY_FEATURES_PATH = (
    PROJECT_DIR
    / "data"
    / "model_ready"
    / "crypto_daily_features_v1.parquet"
)

ROBUSTNESS_ROOT = (
    PROJECT_DIR
    / "data"
    / "model_ready"
    / "rolling30_historical_variance"
)

ROLLING30_DAILY_FEATURES_PATH = (
    ROBUSTNESS_ROOT
    / "crypto_daily_features_rolling30_v1.parquet"
)

ROBUSTNESS_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("Source file:")
print(SOURCE_DAILY_FEATURES_PATH)

print("\nRobustness output root:")
print(ROBUSTNESS_ROOT)

Mounted at /content/drive
Source file:
/content/drive/MyDrive/Quant Research/data/model_ready/crypto_daily_features_v1.parquet

Robustness output root:
/content/drive/MyDrive/Quant Research/data/model_ready/rolling30_historical_variance


## 2. Experiment configuration

In [2]:
EPSILON = 1e-12

ROLLING_WINDOW = 30
ROLLING_DDOF = 1

ASSETS = [
    "BTCUSDT",
    "ETHUSDT",
    "SOLUSDT",
    "XRPUSDT",
]

LOOKBACK_OPTIONS = [
    7,
    14,
    30,
]

TRAIN_END = pd.Timestamp(
    "2024-07-31",
    tz="UTC",
)

VALIDATION_START = pd.Timestamp(
    "2024-08-01",
    tz="UTC",
)

VALIDATION_END = pd.Timestamp(
    "2025-07-31",
    tz="UTC",
)

TEST_START = pd.Timestamp(
    "2025-08-01",
    tz="UTC",
)

TEST_END = pd.Timestamp(
    "2026-07-31",
    tz="UTC",
)

FEATURE_COLUMNS = [
    "log_rolling_30d_variance",
    "daily_return",
    "absolute_daily_return",
    "high_low_range",
    "log_volume",
    "rolling30_var_mean_7d",
    "rolling30_var_mean_30d",
]

TARGET_COLUMN = (
    "target_log_rolling_30d_variance"
)

print("Target:", TARGET_COLUMN)
print("Features:", FEATURE_COLUMNS)

Target: target_log_rolling_30d_variance
Features: ['log_rolling_30d_variance', 'daily_return', 'absolute_daily_return', 'high_low_range', 'log_volume', 'rolling30_var_mean_7d', 'rolling30_var_mean_30d']


## 3. Load the finalized daily feature file

The robustness pipeline deliberately starts from `crypto_daily_features_v1.parquet` rather than rebuilding daily returns. This ensures the alternative target uses exactly the same daily-return observations as the primary experiment.

In [3]:
if not SOURCE_DAILY_FEATURES_PATH.exists():
    raise FileNotFoundError(
        "Finalized daily feature file was not found:\n"
        f"{SOURCE_DAILY_FEATURES_PATH}"
    )

source_df = pd.read_parquet(
    SOURCE_DAILY_FEATURES_PATH
)

required_source_columns = [
    "symbol",
    "date",
    "daily_return",
    "absolute_daily_return",
    "high_low_range",
    "log_volume",
]

missing_columns = [
    col
    for col in required_source_columns
    if col not in source_df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required source columns: "
        f"{missing_columns}"
    )

daily_df = (
    source_df[
        required_source_columns
    ]
    .copy()
)

daily_df["date"] = pd.to_datetime(
    daily_df["date"],
    utc=True,
    errors="raise",
)

daily_df = (
    daily_df
    .loc[
        daily_df["symbol"].isin(
            ASSETS
        )
    ]
    .sort_values(
        [
            "symbol",
            "date",
        ]
    )
    .reset_index(drop=True)
)

duplicate_count = int(
    daily_df.duplicated(
        subset=[
            "symbol",
            "date",
        ]
    ).sum()
)

if duplicate_count:
    raise ValueError(
        f"Found {duplicate_count} duplicate "
        "symbol-date rows."
    )

print("Daily rows:", len(daily_df))

display(
    daily_df
    .groupby("symbol")
    .agg(
        first_date=(
            "date",
            "min",
        ),
        last_date=(
            "date",
            "max",
        ),
        rows=(
            "date",
            "size",
        ),
        non_null_returns=(
            "daily_return",
            "count",
        ),
    )
)

Daily rows: 11738


,first_date,last_date,rows,non_null_returns
symbol,,,,
BTCUSDT,2017-08-17 00:00:00+00:00,2026-08-01 00:00:00+00:00,3272,3271
ETHUSDT,2017-08-17 00:00:00+00:00,2026-08-01 00:00:00+00:00,3272,3271
SOLUSDT,2020-08-11 00:00:00+00:00,2026-08-01 00:00:00+00:00,2182,2181
XRPUSDT,2018-05-04 00:00:00+00:00,2026-08-01 00:00:00+00:00,3012,3011


## 4. Calculate 30-day rolling historical volatility and variance

A rolling observation is valid only if:

- all 30 daily returns are non-null; and
- the 30 rows span exactly 29 calendar days.

This prevents a rolling window from crossing a missing date.

In [5]:
def add_rolling30_target_state(
    asset_df,
):
    out = (
        asset_df
        .sort_values("date")
        .reset_index(drop=True)
        .copy()
    )

    rolling_returns = (
        out["daily_return"]
        .rolling(
            window=ROLLING_WINDOW,
            min_periods=ROLLING_WINDOW,
        )
    )

    rolling_count = (
        rolling_returns.count()
    )

    date_span = (
        out["date"]
        - out["date"].shift(
            ROLLING_WINDOW - 1
        )
    )

    valid_rolling30 = (
        rolling_count.eq(
            ROLLING_WINDOW
        )
        & date_span.eq(
            pd.Timedelta(
                days=ROLLING_WINDOW - 1
            )
        )
    )

    out[
        "rolling_30d_volatility"
    ] = (
        rolling_returns
        .std(ddof=ROLLING_DDOF)
        .where(valid_rolling30)
    )

    out[
        "rolling_30d_variance"
    ] = (
        out[
            "rolling_30d_volatility"
        ] ** 2
    )

    out[
        "log_rolling_30d_variance"
    ] = np.log(
        out[
            "rolling_30d_variance"
        ]
        + EPSILON
    )

    return out


asset_frames = []

for symbol in ASSETS:

    asset_df = (
        daily_df.loc[
            daily_df["symbol"].eq(
                symbol
            )
        ]
        .copy()
    )

    asset_frames.append(
        add_rolling30_target_state(
            asset_df
        )
    )

daily_df = (
    pd.concat(
        asset_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "symbol",
            "date",
        ]
    )
    .reset_index(drop=True)
)

rolling30_summary = (
    daily_df.loc[
        daily_df[
            "rolling_30d_variance"
        ].notna()
    ]
    .groupby("symbol")
    .agg(
        first_valid_date=(
            "date",
            "min",
        ),
        last_valid_date=(
            "date",
            "max",
        ),
        valid_rows=(
            "rolling_30d_variance",
            "count",
        ),
    )
)

display(
    rolling30_summary
)

,first_valid_date,last_valid_date,valid_rows
symbol,,,
BTCUSDT,2017-09-16 00:00:00+00:00,2026-08-01 00:00:00+00:00,3242
ETHUSDT,2017-09-16 00:00:00+00:00,2026-08-01 00:00:00+00:00,3242
SOLUSDT,2020-09-10 00:00:00+00:00,2026-08-01 00:00:00+00:00,2152
XRPUSDT,2018-06-03 00:00:00+00:00,2026-08-01 00:00:00+00:00,2982


## 5. Add rolling means of the new variance state

These are analogous to the `rv_mean_7d` and `rv_mean_30d` features used in the primary pipeline, but are calculated from the new rolling-30 historical variance.

In [6]:
def continuous_rolling_mean(
    asset_df,
    source_column,
    window,
):
    rolling_values = (
        asset_df[source_column]
        .rolling(
            window=window,
            min_periods=window,
        )
    )

    rolling_count = (
        rolling_values.count()
    )

    date_span = (
        asset_df["date"]
        - asset_df["date"].shift(
            window - 1
        )
    )

    valid_window = (
        rolling_count.eq(window)
        & date_span.eq(
            pd.Timedelta(
                days=window - 1
            )
        )
    )

    return (
        rolling_values
        .mean()
        .where(valid_window)
    )


asset_frames = []

for symbol in ASSETS:

    asset_df = (
        daily_df.loc[
            daily_df["symbol"].eq(
                symbol
            )
        ]
        .sort_values("date")
        .reset_index(drop=True)
        .copy()
    )

    asset_df[
        "rolling30_var_mean_7d"
    ] = continuous_rolling_mean(
        asset_df,
        "rolling_30d_variance",
        7,
    )

    asset_df[
        "rolling30_var_mean_30d"
    ] = continuous_rolling_mean(
        asset_df,
        "rolling_30d_variance",
        30,
    )

    asset_frames.append(
        asset_df
    )

daily_df = (
    pd.concat(
        asset_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "symbol",
            "date",
        ]
    )
    .reset_index(drop=True)
)

display(
    daily_df[
        [
            "symbol",
            "date",
            "rolling_30d_volatility",
            "rolling_30d_variance",
            "log_rolling_30d_variance",
            "rolling30_var_mean_7d",
            "rolling30_var_mean_30d",
        ]
    ].tail(10)
)

,symbol,date,rolling_30d_volatility,rolling_30d_variance,log_rolling_30d_variance,rolling30_var_mean_7d,rolling30_var_mean_30d
11728,XRPUSDT,2026-07-23 00:00:00+00:00,0.020166,0.000407,-7.807546,0.000387,0.000594
11729,XRPUSDT,2026-07-24 00:00:00+00:00,0.019352,0.000374,-7.889944,0.000382,0.000580
11730,XRPUSDT,2026-07-25 00:00:00+00:00,0.018573,0.000345,-7.972098,0.000378,0.000564
11731,XRPUSDT,2026-07-26 00:00:00+00:00,0.018698,0.000350,-7.958726,0.000374,0.000548
11732,XRPUSDT,2026-07-27 00:00:00+00:00,0.020430,0.000417,-7.781478,0.000380,0.000535
11733,XRPUSDT,2026-07-28 00:00:00+00:00,0.020434,0.000418,-7.781123,0.000385,0.000521
11734,XRPUSDT,2026-07-29 00:00:00+00:00,0.020371,0.000415,-7.787243,0.000389,0.000508
11735,XRPUSDT,2026-07-30 00:00:00+00:00,0.020116,0.000405,-7.812433,0.000389,0.000495
11736,XRPUSDT,2026-07-31 00:00:00+00:00,0.020358,0.000414,-7.788571,0.000395,0.000481
11737,XRPUSDT,2026-08-01 00:00:00+00:00,0.019410,0.000377,-7.883914,0.000399,0.000469


## 6. Create the next-day target

Features are observed through day `t`.  
The target is the rolling-30 historical variance state on calendar day `t+1`.

In [7]:
target_lookup = (
    daily_df[
        [
            "symbol",
            "date",
            "rolling_30d_volatility",
            "rolling_30d_variance",
            "log_rolling_30d_variance",
        ]
    ]
    .rename(
        columns={
            "date":
                "target_date",

            "rolling_30d_volatility":
                "target_rolling_30d_volatility",

            "rolling_30d_variance":
                "target_rolling_30d_variance",

            "log_rolling_30d_variance":
                "target_log_rolling_30d_variance",
        }
    )
)

daily_df["target_date"] = (
    daily_df["date"]
    + pd.Timedelta(days=1)
)

daily_df = daily_df.merge(
    target_lookup,
    on=[
        "symbol",
        "target_date",
    ],
    how="left",
    validate="one_to_one",
)

valid_target_rows = (
    daily_df[
        TARGET_COLUMN
    ].notna()
)

if valid_target_rows.any():

    target_gap = (
        daily_df.loc[
            valid_target_rows,
            "target_date",
        ]
        - daily_df.loc[
            valid_target_rows,
            "date",
        ]
    )

    assert (
        target_gap
        .eq(
            pd.Timedelta(days=1)
        )
        .all()
    )

print(
    "Rows with valid next-day target:",
    int(
        valid_target_rows.sum()
    ),
)

print(
    "Rows without valid next-day target:",
    int(
        (~valid_target_rows).sum()
    ),
)

Rows with valid next-day target: 11618
Rows without valid next-day target: 120


## 7. Assign train / validation / test by target date

In [11]:
def assign_split(
    target_date,
):
    if pd.isna(target_date):
        return "outside"

    if target_date <= TRAIN_END:
        return "train"

    if (
        VALIDATION_START
        <= target_date
        <= VALIDATION_END
    ):
        return "validation"

    if (
        TEST_START
        <= target_date
        <= TEST_END
    ):
        return "test"

    return "outside"


daily_df["split"] = (
    daily_df[
        "target_date"
    ]
    .apply(assign_split)
)

split_summary = (
    daily_df.loc[
        daily_df[
            TARGET_COLUMN
        ].notna()
    ]
    .groupby("split")
    .agg(
        start_target_date=(
            "target_date",
            "min",
        ),
        end_target_date=(
            "target_date",
            "max",
        ),
        rows=(
            "target_date",
            "count",
        ),
    )
)

display(
    split_summary
)

for expected_split in [
    "train",
    "validation",
    "test",
]:
    if (
        expected_split
        not in split_summary.index
    ):
        raise ValueError(
            f"Missing split: "
            f"{expected_split}"
        )

assert (
    split_summary.loc[
        "validation",
        "start_target_date",
    ]
    == VALIDATION_START
)

assert (
    split_summary.loc[
        "validation",
        "end_target_date",
    ]
    == VALIDATION_END
)

assert (
    split_summary.loc[
        "test",
        "start_target_date",
    ]
    == TEST_START
)

assert (
    split_summary.loc[
        "test",
        "end_target_date",
    ]
    == TEST_END
)

,start_target_date,end_target_date,rows
split,,,
outside,2026-08-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,4
test,2025-08-01 00:00:00+00:00,2026-07-31 00:00:00+00:00,1460
train,2017-09-16 00:00:00+00:00,2024-07-31 00:00:00+00:00,8694
validation,2024-08-01 00:00:00+00:00,2025-07-31 00:00:00+00:00,1460


## 8. Save the all-asset rolling-30 daily feature dataset

This file will be the source for the later global multi-asset robustness models and the ARCH robustness notebook.

In [12]:
output_columns = [
    "symbol",
    "date",
    "daily_return",
    "absolute_daily_return",
    "high_low_range",
    "log_volume",
    "rolling_30d_volatility",
    "rolling_30d_variance",
    "log_rolling_30d_variance",
    "rolling30_var_mean_7d",
    "rolling30_var_mean_30d",
    "target_date",
    "target_rolling_30d_volatility",
    "target_rolling_30d_variance",
    "target_log_rolling_30d_variance",
    "split",
]

rolling30_daily_df = (
    daily_df[
        output_columns
    ]
    .copy()
)

rolling30_daily_df.to_parquet(
    ROLLING30_DAILY_FEATURES_PATH,
    index=False,
)

print(
    "Saved:",
    ROLLING30_DAILY_FEATURES_PATH,
)

print(
    "Shape:",
    rolling30_daily_df.shape,
)

Saved: /content/drive/MyDrive/Quant Research/data/model_ready/rolling30_historical_variance/crypto_daily_features_rolling30_v1.parquet
Shape: (11738, 16)


## 9. Create leakage-safe BTC sequences

The sequence logic mirrors the primary pipeline:

- feature window must contain consecutive calendar days;
- no feature value may be missing;
- target must be exactly one calendar day after the sequence end;
- target split determines train / validation / test.

In [13]:
def create_sequences(
    asset_df,
    feature_columns,
    lookback_days,
):
    asset_df = (
        asset_df
        .sort_values("date")
        .reset_index(drop=True)
        .copy()
    )

    X = []
    y = []
    metadata = []

    feature_values = (
        asset_df[
            feature_columns
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    target_values = (
        asset_df[
            TARGET_COLUMN
        ]
        .to_numpy(
            dtype=np.float32
        )
    )

    for end_idx in range(
        lookback_days - 1,
        len(asset_df),
    ):

        start_idx = (
            end_idx
            - lookback_days
            + 1
        )

        window = asset_df.iloc[
            start_idx:
            end_idx + 1
        ]

        final_row = (
            asset_df.iloc[
                end_idx
            ]
        )

        dates = window["date"]

        if (
            dates.iloc[-1]
            - dates.iloc[0]
            != pd.Timedelta(
                days=lookback_days - 1
            )
        ):
            continue

        if not (
            dates.diff()
            .dropna()
            .eq(
                pd.Timedelta(days=1)
            )
            .all()
        ):
            continue

        sequence = (
            feature_values[
                start_idx:
                end_idx + 1
            ]
        )

        if not np.isfinite(
            sequence
        ).all():
            continue

        target_value = (
            target_values[
                end_idx
            ]
        )

        if not np.isfinite(
            target_value
        ):
            continue

        if (
            final_row[
                "target_date"
            ]
            - final_row["date"]
            != pd.Timedelta(days=1)
        ):
            continue

        if (
            final_row["split"]
            not in {
                "train",
                "validation",
                "test",
            }
        ):
            continue

        X.append(
            sequence
        )

        y.append(
            target_value
        )

        metadata.append({
            "symbol":
                final_row["symbol"],

            "sequence_start_date":
                dates.iloc[0],

            "sequence_end_date":
                final_row["date"],

            "target_date":
                final_row[
                    "target_date"
                ],

            "actual_log_rolling30_variance":
                final_row[
                    "target_log_rolling_30d_variance"
                ],

            "actual_rolling30_variance":
                final_row[
                    "target_rolling_30d_variance"
                ],

            "actual_rolling30_volatility":
                final_row[
                    "target_rolling_30d_volatility"
                ],

            "split":
                final_row["split"],
        })

    return (
        np.asarray(
            X,
            dtype=np.float32,
        ),
        np.asarray(
            y,
            dtype=np.float32,
        ),
        pd.DataFrame(
            metadata
        ),
    )

## 10. Build and save BTC 7d / 14d / 30d artifacts

The scaler is fitted only on training-sequence feature values.  
Validation and test arrays are transformed using that same training scaler.

In [14]:
btc_daily_df = (
    rolling30_daily_df.loc[
        rolling30_daily_df[
            "symbol"
        ].eq("BTCUSDT")
    ]
    .copy()
)

artifact_summary_rows = []

for lookback_days in (
    LOOKBACK_OPTIONS
):

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"BTCUSDT — "
        f"{lookback_days}d lookback"
    )

    print(
        "=" * 70
    )

    (
        X_raw,
        y_raw,
        sequence_metadata,
    ) = create_sequences(
        btc_daily_df,
        FEATURE_COLUMNS,
        lookback_days,
    )

    if sequence_metadata.empty:
        raise ValueError(
            "No valid sequences were "
            f"created for {lookback_days}d."
        )

    train_mask = (
        sequence_metadata[
            "split"
        ]
        .eq("train")
        .to_numpy()
    )

    validation_mask = (
        sequence_metadata[
            "split"
        ]
        .eq("validation")
        .to_numpy()
    )

    test_mask = (
        sequence_metadata[
            "split"
        ]
        .eq("test")
        .to_numpy()
    )

    X_train_raw = X_raw[
        train_mask
    ]

    X_val_raw = X_raw[
        validation_mask
    ]

    X_test_raw = X_raw[
        test_mask
    ]

    y_train = y_raw[
        train_mask
    ]

    y_val = y_raw[
        validation_mask
    ]

    y_test = y_raw[
        test_mask
    ]

    train_metadata = (
        sequence_metadata.loc[
            train_mask
        ]
        .reset_index(drop=True)
    )

    validation_metadata = (
        sequence_metadata.loc[
            validation_mask
        ]
        .reset_index(drop=True)
    )

    test_metadata = (
        sequence_metadata.loc[
            test_mask
        ]
        .reset_index(drop=True)
    )

    if len(X_train_raw) == 0:
        raise ValueError(
            "Training sequences are empty."
        )

    n_features = (
        X_train_raw.shape[2]
    )

    scaler = (
        StandardScaler()
        .fit(
            X_train_raw.reshape(
                -1,
                n_features,
            )
        )
    )

    def scale_sequences(
        X,
    ):
        return (
            scaler
            .transform(
                X.reshape(
                    -1,
                    n_features,
                )
            )
            .reshape(
                X.shape
            )
            .astype(
                np.float32
            )
        )

    X_train_scaled = (
        scale_sequences(
            X_train_raw
        )
    )

    X_val_scaled = (
        scale_sequences(
            X_val_raw
        )
    )

    X_test_scaled = (
        scale_sequences(
            X_test_raw
        )
    )

    output_dir = (
        ROBUSTNESS_ROOT
        / "BTCUSDT"
        / (
            f"lookback_"
            f"{lookback_days}d"
        )
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    arrays_to_save = {
        "X_train_raw.npy":
            X_train_raw,

        "X_val_raw.npy":
            X_val_raw,

        "X_test_raw.npy":
            X_test_raw,

        "X_train_scaled.npy":
            X_train_scaled,

        "X_val_scaled.npy":
            X_val_scaled,

        "X_test_scaled.npy":
            X_test_scaled,

        "y_train.npy":
            y_train,

        "y_val.npy":
            y_val,

        "y_test.npy":
            y_test,
    }

    for (
        filename,
        array,
    ) in arrays_to_save.items():

        np.save(
            output_dir
            / filename,
            array,
        )

    train_metadata.to_parquet(
        output_dir
        / "train_metadata.parquet",
        index=False,
    )

    validation_metadata.to_parquet(
        output_dir
        / "validation_metadata.parquet",
        index=False,
    )

    test_metadata.to_parquet(
        output_dir
        / "test_metadata.parquet",
        index=False,
    )

    joblib.dump(
        scaler,
        output_dir
        / "feature_scaler.joblib",
    )

    configuration = {
        "asset":
            "BTCUSDT",

        "candle_interval":
            "1d",

        "forecast_horizon_days":
            1,

        "lookback_days":
            lookback_days,

        "volatility_definition":
            "30_day_rolling_std_of_daily_log_returns",

        "variance_definition":
            "square_of_30_day_rolling_std_of_daily_log_returns",

        "rolling_window_days":
            ROLLING_WINDOW,

        "rolling_std_ddof":
            ROLLING_DDOF,

        "target":
            "log_rolling_30d_variance",

        "target_column":
            TARGET_COLUMN,

        "feature_columns":
            FEATURE_COLUMNS,

        "train_end":
            str(TRAIN_END),

        "validation_start":
            str(
                VALIDATION_START
            ),

        "validation_end":
            str(
                VALIDATION_END
            ),

        "test_start":
            str(
                TEST_START
            ),

        "test_end":
            str(
                TEST_END
            ),

        "X_train_shape":
            list(
                X_train_scaled.shape
            ),

        "X_validation_shape":
            list(
                X_val_scaled.shape
            ),

        "X_test_shape":
            list(
                X_test_scaled.shape
            ),
    }

    with open(
        output_dir
        / "dataset_configuration.json",
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            configuration,
            file,
            indent=2,
        )

    print(
        "Train:",
        X_train_scaled.shape,
        train_metadata[
            "target_date"
        ].min(),
        "to",
        train_metadata[
            "target_date"
        ].max(),
    )

    print(
        "Validation:",
        X_val_scaled.shape,
        validation_metadata[
            "target_date"
        ].min(),
        "to",
        validation_metadata[
            "target_date"
        ].max(),
    )

    print(
        "Test:",
        X_test_scaled.shape,
        test_metadata[
            "target_date"
        ].min(),
        "to",
        test_metadata[
            "target_date"
        ].max(),
    )

    assert (
        validation_metadata[
            "target_date"
        ].min()
        == VALIDATION_START
    )

    assert (
        validation_metadata[
            "target_date"
        ].max()
        == VALIDATION_END
    )

    assert (
        test_metadata[
            "target_date"
        ].min()
        == TEST_START
    )

    assert (
        test_metadata[
            "target_date"
        ].max()
        == TEST_END
    )

    artifact_summary_rows.append({
        "lookback_days":
            lookback_days,

        "train_sequences":
            len(X_train_scaled),

        "validation_sequences":
            len(X_val_scaled),

        "test_sequences":
            len(X_test_scaled),

        "n_features":
            n_features,

        "output_dir":
            str(output_dir),
    })


artifact_summary_df = pd.DataFrame(
    artifact_summary_rows
)

display(
    artifact_summary_df
)


BTCUSDT — 7d lookback
Train: (2475, 7, 7) 2017-10-22 00:00:00+00:00 to 2024-07-31 00:00:00+00:00
Validation: (365, 7, 7) 2024-08-01 00:00:00+00:00 to 2025-07-31 00:00:00+00:00
Test: (365, 7, 7) 2025-08-01 00:00:00+00:00 to 2026-07-31 00:00:00+00:00

BTCUSDT — 14d lookback
Train: (2468, 14, 7) 2017-10-29 00:00:00+00:00 to 2024-07-31 00:00:00+00:00
Validation: (365, 14, 7) 2024-08-01 00:00:00+00:00 to 2025-07-31 00:00:00+00:00
Test: (365, 14, 7) 2025-08-01 00:00:00+00:00 to 2026-07-31 00:00:00+00:00

BTCUSDT — 30d lookback
Train: (2452, 30, 7) 2017-11-14 00:00:00+00:00 to 2024-07-31 00:00:00+00:00
Validation: (365, 30, 7) 2024-08-01 00:00:00+00:00 to 2025-07-31 00:00:00+00:00
Test: (365, 30, 7) 2025-08-01 00:00:00+00:00 to 2026-07-31 00:00:00+00:00


,lookback_days,train_sequences,validation_sequences,test_sequences,n_features,output_dir
0,7,2475,365,365,7,/content/drive/MyDrive/Quant Research/data/mod...
1,14,2468,365,365,7,/content/drive/MyDrive/Quant Research/data/mod...
2,30,2452,365,365,7,/content/drive/MyDrive/Quant Research/data/mod...


## 11. Save robustness dataset manifest

In [15]:
manifest = {
    "dataset":
        "crypto_daily_features_rolling30_v1",

    "source_daily_features":
        str(
            SOURCE_DAILY_FEATURES_PATH
        ),

    "output_daily_features":
        str(
            ROLLING30_DAILY_FEATURES_PATH
        ),

    "assets":
        ASSETS,

    "target_volatility_definition":
        "30-day rolling sample standard deviation of daily log returns",

    "target_variance_definition":
        "square of 30-day rolling sample standard deviation of daily log returns",

    "rolling_window_days":
        ROLLING_WINDOW,

    "rolling_std_ddof":
        ROLLING_DDOF,

    "model_target":
        "log_rolling_30d_variance",

    "forecast_horizon_days":
        1,

    "feature_columns":
        FEATURE_COLUMNS,

    "lookback_options":
        LOOKBACK_OPTIONS,

    "train_end":
        str(TRAIN_END),

    "validation_start":
        str(
            VALIDATION_START
        ),

    "validation_end":
        str(
            VALIDATION_END
        ),

    "test_start":
        str(
            TEST_START
        ),

    "test_end":
        str(
            TEST_END
        ),

    "total_daily_rows":
        int(
            len(
                rolling30_daily_df
            )
        ),

    "valid_target_rows":
        int(
            rolling30_daily_df[
                TARGET_COLUMN
            ]
            .notna()
            .sum()
        ),
}

manifest_path = (
    ROBUSTNESS_ROOT
    / "rolling30_dataset_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        manifest,
        file,
        indent=2,
    )

print(
    "Saved:",
    manifest_path,
)

Saved: /content/drive/MyDrive/Quant Research/data/model_ready/rolling30_historical_variance/rolling30_dataset_manifest.json


## 12. Final reload and integrity checks

This cell verifies the files that later ARCH and DL notebooks will actually read.

In [16]:
reloaded_daily_df = (
    pd.read_parquet(
        ROLLING30_DAILY_FEATURES_PATH
    )
)

assert (
    reloaded_daily_df.duplicated(
        subset=[
            "symbol",
            "date",
        ]
    ).sum()
    == 0
)

assert set(
    FEATURE_COLUMNS
).issubset(
    reloaded_daily_df.columns
)

assert (
    TARGET_COLUMN
    in reloaded_daily_df.columns
)

for lookback_days in (
    LOOKBACK_OPTIONS
):

    output_dir = (
        ROBUSTNESS_ROOT
        / "BTCUSDT"
        / (
            f"lookback_"
            f"{lookback_days}d"
        )
    )

    X_train = np.load(
        output_dir
        / "X_train_scaled.npy"
    )

    X_val = np.load(
        output_dir
        / "X_val_scaled.npy"
    )

    y_train = np.load(
        output_dir
        / "y_train.npy"
    )

    y_val = np.load(
        output_dir
        / "y_val.npy"
    )

    validation_metadata = (
        pd.read_parquet(
            output_dir
            / "validation_metadata.parquet"
        )
    )

    with open(
        output_dir
        / "dataset_configuration.json",
        "r",
        encoding="utf-8",
    ) as file:

        config = json.load(
            file
        )

    assert np.isfinite(
        X_train
    ).all()

    assert np.isfinite(
        X_val
    ).all()

    assert np.isfinite(
        y_train
    ).all()

    assert np.isfinite(
        y_val
    ).all()

    assert (
        len(X_train)
        == len(y_train)
    )

    assert (
        len(X_val)
        == len(y_val)
    )

    assert (
        len(X_val)
        == len(
            validation_metadata
        )
    )

    assert (
        X_train.shape[1]
        == lookback_days
    )

    assert (
        X_train.shape[2]
        == len(
            FEATURE_COLUMNS
        )
    )

    assert (
        config[
            "candle_interval"
        ]
        == "1d"
    )

    assert (
        config[
            "target"
        ]
        == "log_rolling_30d_variance"
    )

    assert (
        config[
            "variance_definition"
        ]
        == "square_of_30_day_rolling_std_of_daily_log_returns"
    )

    print(
        f"{lookback_days}d OK | "
        f"train={X_train.shape} | "
        f"validation={X_val.shape}"
    )

print(
    "\nAll rolling-30 robustness "
    "artifacts passed reload checks."
)

7d OK | train=(2475, 7, 7) | validation=(365, 7, 7)
14d OK | train=(2468, 14, 7) | validation=(365, 14, 7)
30d OK | train=(2452, 30, 7) | validation=(365, 30, 7)

All rolling-30 robustness artifacts passed reload checks.


## Expected outputs

After a successful run, the important files will be:

```text
Quant Research/
└── data/
    └── model_ready/
        └── rolling30_historical_variance/
            ├── crypto_daily_features_rolling30_v1.parquet
            ├── rolling30_dataset_manifest.json
            └── BTCUSDT/
                ├── lookback_7d/
                ├── lookback_14d/
                └── lookback_30d/
```

Each BTC lookback folder contains the same artifact types used by the primary DL pipeline:

- raw and scaled train / validation / test arrays;
- log-variance targets;
- train / validation / test metadata;
- training-fitted feature scaler;
- dataset configuration JSON.

The original `crypto_daily_features_v1.parquet` and primary model-ready artifacts are untouched.